# Notebook 03 — Streaming Responses & Frontend Integration

## Objectives
- Understand generator-based token streaming in Python
- Learn Server-Sent Events (SSE) format for real-time web responses
- See the FastAPI `StreamingResponse` pattern for SSE
- Understand Vercel AI SDK `useChat` React hook integration
- Use `day5.streaming` for production-ready streaming

## 1. Generator-based token streaming (pure Python)

In [ ]:
import time

def stream_text(text, delay=0):
    """Yield tokens one by one — this is how LLM streaming works."""
    for word in text.split():
        yield word + " "
        # time.sleep(delay)  # uncomment for real delay

answer = "LangGraph enables stateful multi-agent orchestration with streaming support"
print("Streaming output:")
for token in stream_text(answer):
    print(token, end="", flush=True)
print("\nDone!")

## Server-Sent Events (SSE) Format

SSE is a simple HTTP protocol for one-way server → client streaming:

```
data: {"type": "token", "content": "Hello "}

data: {"type": "token", "content": "world!"}

data: {"type": "done", "content": "", "metadata": {"tokens": 2}}

```

Key rules:
- Each message starts with `data: `
- Messages are separated by **two** newlines (`\n\n`)
- The browser's `EventSource` API reads these automatically
- FastAPI's `StreamingResponse` with `media_type="text/event-stream"` handles the HTTP headers

## 2. SSE format inline

In [ ]:
import json

def to_sse(data: dict) -> str:
    return f"data: {json.dumps(data)}\n\n"

# Simulate streaming response
chunks = [
    {"type": "token",  "content": "Hybrid "},
    {"type": "token",  "content": "search "},
    {"type": "token",  "content": "combines BM25 and semantic methods."},
    {"type": "source", "content": "hybrid_search.md"},
    {"type": "done",   "content": "", "metadata": {"tokens": 8}},
]
for c in chunks:
    print(repr(to_sse(c)))

## 3. FastAPI StreamingResponse pattern

In [ ]:
fastapi_code = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import json

app = FastAPI()

@app.get("/stream")
async def stream_answer(query: str):
    """Stream answer tokens via Server-Sent Events."""
    def generate():
        # In production: run your orchestrator here
        answer = f"The answer to '{query}' is: ..."
        for word in answer.split():
            yield f"data: {json.dumps({'type': 'token', 'content': word + ' '})}\\n\\n"
        yield f"data: {json.dumps({'type': 'done', 'content': ''})}\\n\\n"
    
    return StreamingResponse(
        generate(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"}
    )
'''
print(fastapi_code)

## 4. Vercel AI SDK / React frontend

The Vercel AI SDK's `useChat` hook connects a React frontend to any SSE endpoint:

In [ ]:
react_code = '''
// React component using Vercel AI SDK useChat hook
import { useChat } from "ai/react";

export default function ChatUI() {
  const { messages, input, handleInputChange, handleSubmit } = useChat({
    api: "http://localhost:8000/stream",  // Points to FastAPI SSE endpoint
  });

  return (
    <div>
      {messages.map(m => (
        <div key={m.id}>
          <strong>{m.role}:</strong> {m.content}
        </div>
      ))}
      <form onSubmit={handleSubmit}>
        <input value={input} onChange={handleInputChange} />
        <button type="submit">Send</button>
      </form>
    </div>
  );
}
'''
print(react_code)

## 5. Setup sys.path

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 6. Using day5.streaming

In [ ]:
from day5.streaming import StreamChunk, stream_tokens, collect_stream

answer = "Hybrid search combines BM25 keyword retrieval with dense semantic embeddings"
chunks = list(stream_tokens(answer))

print(f"Total chunks: {len(chunks)}")
print(f"Token chunks: {sum(1 for c in chunks if c.chunk_type == 'token')}")
print(f"Done chunk  : {sum(1 for c in chunks if c.chunk_type == 'done')}")
print()

# Show first 3 SSE strings
print("First 3 SSE chunks:")
for c in chunks[:3]:
    print(repr(c.to_sse()))

# Collect back to full text
result = collect_stream(iter(chunks))
print("\nReassembled:", result["answer"])

## 7. Full streaming orchestrator demo

In [ ]:
from day5.streaming import StreamingOrchestrator, fastapi_streaming_example, vercel_ai_sdk_example
from day5.multi_agent_orchestrator import build_orchestrator

docs = [
    "LangGraph enables stateful multi-agent workflows",
    "BM25 is a keyword ranking algorithm for information retrieval",
    "Hybrid search combines BM25 and semantic retrieval methods",
]

def retrieve(q):
    return [d for d in docs if any(w in d.lower() for w in q.lower().split())][:2]

graph = build_orchestrator(retrieve_fn=retrieve)
orch  = StreamingOrchestrator(graph)

print("Streaming response (first 5 chunks):")
chunks = list(orch.stream("hybrid search"))
for c in chunks[:5]:
    print(repr(c))

print(f"\nTotal SSE chunks: {len(chunks)}")

print("\n--- FastAPI pattern ---")
print(fastapi_streaming_example()[:400])